# Healthcare Need Score Analysis

Calculate healthcare need scores for Maricopa County ZCTAs using standardized data.

**Input:** Standardized data from `data_cleaning.ipynb` pipeline  
**Output:** Healthcare need scores ready for dashboard visualization

## Need Score Approach
Using standardized variables (Z_ prefix, 0-1 scale) where:
- **Higher values (closer to 1) = worse outcomes/greater need**
- **Lower values (closer to 0) = better outcomes/lower need**

## Sample Need Score Formula
Weighted composite combining key equity indicators:
- Income: Lower income → higher need
- Healthcare access: Poor access → higher need  
- Chronic disease burden: Higher prevalence → higher need

## 1. Load Standardized Data and Calculate Need Score

In [10]:
# Import libraries and load standardized data
import pandas as pd
import numpy as np
import os

# Load standardized data (already cleaned and processed)
print("Loading standardized data...")
df = pd.read_csv("data/maricopa_healthcare_standardized_data.csv")

print(f"✓ Data loaded: {df.shape[0]} ZCTAs, {df.shape[1]} variables")
print(f"✓ Standardized variables available: {len([col for col in df.columns if col.startswith('Z_')])}")

Loading standardized data...
✓ Data loaded: 128 ZCTAs, 52 variables
✓ Standardized variables available: 25


### Test--Ainsley you can delete this

In [11]:
# Calculate healthcare equity need score using weighted composite approach
print("Calculating healthcare equity need score...")

# Key variables for need score (already standardized 0-1 where 1=worse outcomes)
# Income is already flipped in cleaning: higher income → lower Z_census_B19013_001E
# Access variables are flipped: more providers/checkups → lower Z_ values
# Chronic disease variables: higher prevalence → higher Z_ values

# Weighted composite score
df["need_score"] = (
    0.4 * df["Z_census_B19013_001E"] +      # Income (40% weight) - already flipped
    0.3 * df["Z_cdc_diabetes_crudeprev"] +  # Diabetes prevalence (30% weight)
    0.2 * df["Z_npi_total_providers"] +     # Provider access (20% weight) - already flipped  
    0.1 * df["Z_cdc_access2_crudeprev"]     # Lack of insurance (10% weight)
)

print("✓ Need score calculated: need_score")
print(f"  Range: {df['need_score'].min():.3f} to {df['need_score'].max():.3f}")
print(f"  Mean: {df['need_score'].mean():.3f}")
print(f"  Higher scores = greater healthcare equity needs")

# Show sample data
print(f"\nSample of ZCTAs with need scores:")
display(df[['zcta', 'need_score', 'Z_census_B19013_001E', 'Z_cdc_diabetes_crudeprev', 'Z_npi_total_providers']].head())

Calculating healthcare equity need score...
✓ Need score calculated: need_score
  Range: 0.197 to 0.940
  Mean: 0.498
  Higher scores = greater healthcare equity needs

Sample of ZCTAs with need scores:


,zcta,need_score,Z_census_B19013_001E,Z_cdc_diabetes_crudeprev,Z_npi_total_providers
0,85003,0.624072,0.931343,0.364341,0.513308
1,85004,0.530438,0.823140,0.310078,0.346008
2,85006,0.644627,0.901134,0.620155,0.110266
3,85007,0.735929,0.937571,0.736434,0.399240
4,85008,0.591121,0.896480,0.387597,0.277567


## Need Score Edit

In [12]:
# Calculate healthcare equity need score using weighted composite approach
print("Calculating healthcare equity need score...")

# Key variables for need score (already standardized 0-1 where 1=worse outcomes)
# Income is already flipped in cleaning: higher income → lower Z_census_B19013_001E
# Access variables are flipped: more providers/checkups → lower Z_ values
# Chronic disease variables: higher prevalence → higher Z_ values

#Combine chronic disease variables into one average score
df["Z_chronic_disease_avg"] = df[["Z_cdc_diabetes_crudeprev", 
                                  "Z_cdc_highchol_crudeprev",
                                  "Z_cdc_chd_crudeprev",
                                  "Z_cdc_casthma_crudeprev",
                                  "Z_cdc_copd_crudeprev",
                                  "Z_cdc_obesity_crudeprev"]].mean(axis=1)
print("✓ Chronic disease average calculated: Z_chronic_disease_avg")

# Weighted composite score
df["need_score"] = (
    0.2 * df["Z_census_B19013_001E"] +      # Income - already flipped (lower income = higher need)
    0.2 * df["Z_census_B01001_001E"] +      # Population (higher population = higher need)
    0.2 * df["Z_chronic_disease_avg"] +      # Chronic disease prevalence (higher prevalence = higher need)
    0.2 * df["Z_npi_total_providers"] +     # Provider access - already flipped (more providers = lower need)
    0.2 * df["Z_cdc_access2_crudeprev"]     # Lack of insurance (higher lack = higher need)
)

print("✓ Need score calculated: need_score")
print(f"  Range: {df['need_score'].min():.3f} to {df['need_score'].max():.3f}")
print(f"  Mean: {df['need_score'].mean():.3f}")
print(f"  Higher scores = greater healthcare equity needs")

# Show sample data
print(f"\nSample of ZCTAs with need scores:")
display(df[['zcta', 'need_score', 'Z_census_B19013_001E', 'Z_census_B01001_001E', 'Z_chronic_disease_avg', 'Z_npi_total_providers']].head())

Calculating healthcare equity need score...
✓ Chronic disease average calculated: Z_chronic_disease_avg
✓ Need score calculated: need_score
  Range: 0.195 to 0.696
  Mean: 0.428
  Higher scores = greater healthcare equity needs

Sample of ZCTAs with need scores:


,zcta,need_score,Z_census_B19013_001E,Z_census_B01001_001E,Z_chronic_disease_avg,Z_npi_total_providers
0,85003,0.444634,0.931343,0.091492,0.291323,0.513308
1,85004,0.393145,0.823140,0.103555,0.303453,0.346008
2,85006,0.496585,0.901134,0.232121,0.478667,0.110266
3,85007,0.523751,0.937571,0.131584,0.549135,0.399240
4,85008,0.562510,0.896480,0.685136,0.346004,0.277567


## 2. Analyze Need Score Distribution and Identify High-Need Areas
Can also delete

In [13]:
# Analyze need score distribution
print("Need Score Analysis")
print("=" * 30)

# Summary statistics
stats = df['need_score'].describe()
print("Summary Statistics:")
for stat, value in stats.items():
    print(f"  {stat}: {value:.4f}")

# Identify high-need areas (top 25%)
high_need_threshold = df['need_score'].quantile(0.75)
high_need_zctas = df[df['need_score'] >= high_need_threshold].sort_values('need_score', ascending=False)

print(f"\nHigh-Need Areas (75th percentile and above):")
print(f"  Threshold: {high_need_threshold:.4f}")
print(f"  Number of high-need ZCTAs: {len(high_need_zctas)}")

print(f"\nTop 10 Highest Need ZCTAs:")
top_need = high_need_zctas.head(10)[['zcta', 'need_score']].copy()
for _, row in top_need.iterrows():
    print(f"  ZCTA {int(row['zcta'])}: {row['need_score']:.4f}")

# Skip visualization - focus on data processing for dashboard

print(f"\n✓ Analysis complete - {len(high_need_zctas)} high-need areas identified")

Need Score Analysis
Summary Statistics:
  count: 128.0000
  mean: 0.4281
  std: 0.1095
  min: 0.1953
  25%: 0.3558
  50%: 0.4177
  75%: 0.5113
  max: 0.6960

High-Need Areas (75th percentile and above):
  Threshold: 0.5113
  Number of high-need ZCTAs: 32

Top 10 Highest Need ZCTAs:
  ZCTA 85301: 0.6960
  ZCTA 85337: 0.6866
  ZCTA 85009: 0.6865
  ZCTA 85035: 0.6604
  ZCTA 85033: 0.6563
  ZCTA 85017: 0.6302
  ZCTA 85041: 0.6173
  ZCTA 85031: 0.6100
  ZCTA 85321: 0.6028
  ZCTA 85019: 0.6021

✓ Analysis complete - 32 high-need areas identified


## 3. Export Data with Need Scores for Dashboard

In [14]:
# Export complete dataset with need scores for dashboard
print("Exporting data with need scores...")

# Ensure data directory exists
os.makedirs("data", exist_ok=True)

# Export complete dataset with need score
output_file = "data/maricopa_healthcare_standardized_needscore.csv"
df.to_csv(output_file, index=False)

print(f"✓ Complete dataset exported: {output_file}")
print(f"  Shape: {df.shape}")
print(f"  Includes: All original + standardized variables + need_score")
print(f"  Ready for dashboard integration and analysis")

Exporting data with need scores...
✓ Complete dataset exported: data/maricopa_healthcare_standardized_needscore.csv
  Shape: (128, 54)
  Includes: All original + standardized variables + need_score
  Ready for dashboard integration and analysis
